# Ventricular tachyarrhythmias: detection pipeline

Working notebook for the paper's experiments, modeled on `vfpred/codes/VFPred.ipynb`.
It runs our steps end to end and holds results as each phase lands. See `paper/PLAN.md`
for the full plan and `paper/PAPER.md` for the manuscript.

Unlike VFPred, this pipeline is classical and deterministic: no SVM, no CNN. The candidate
detectors are TCSC, VFLEAK, SPEC, HILB, and MEA, compared on shockable vs non-shockable and
judged on discrimination and compute cost.

Dataset decisions in force (PLAN.md, Phase 2): read through `pxg.cbor`, filter once per
record, slide overlapping 1 s-step windows at 8 s and 4 s, write per-record TSVs under
`data/s<sec>/`. Feature extraction feeds vftx a millivolt signal (`int / 200`) with vftx's
own filtering off (`apply_filters=False`); QRS features are skipped until the exg-core beat
annotations are wired in.

## Setup

In [1]:
import os

# This notebook lives in vfta/. Run everything from the repo root so `from vfta import ...`
# and the relative data/ and work/ (cbor) paths resolve. pxg is installed in the conda env.
if os.path.basename(os.getcwd()) == 'vfta':
    os.chdir('..')
pass #if

import numpy as np
import pandas as pd

from vfta import build, labels

DBS = ['mitdb', 'cudb', 'vfdb', 'ahadb']
WINDOWS = [8.0, 4.0]

## Phase 2: build the dataset (per-record TSV)

`cbor record -> record-level filtering -> sliding window -> data/s<sec>/<db>/<rid>.tsv`,
parallel by record. The build is heavy, so run it once per database and window; the cells
below are left commented to avoid rerunning by accident.

In [2]:
# for win in WINDOWS:
#     for db in DBS:
#         res = build.build_database(db, window_sec=win, step_sec=1.0, jobs=-1)
#         print(f'{db} s{int(win)}: {len(res)} records, {sum(n for _, n in res)} windows')
#     pass #for
# pass #for

## Phase 2: label windows (shockable / VT / VFL / VF)

Each window gets a `Rhythm` from its dominant clean episode (>= 90% of the window) and a
binary `Shock` label. Windows below the purity threshold are `MIX` (evaluation only).

In [3]:
df = labels.load_dataset(['vfdb', 'cudb'], window_sec=8.0, purity=0.9)
print(df['Rhythm'].value_counts())
print(df['Shock'].value_counts())

Rhythm
OTHER    20978
NSR      17375
MIX      14870
VF        5118
VT        4849
VFL        369
Name: count, dtype: int64
Shock
NON      50849
SHOCK    10348
MIX       2362
Name: count, dtype: int64


## Phase 2: feature extraction (vftx)

`build_record` / `build_database` now compute the vftx feature columns per window. Two
families are excluded by design (see `paper/PLAN.md`):

- QRS features (RR, beat ratios): a QRS detector must blank during VF/VFL, so a signal-only
  VF/VFL detector cannot depend on it without circularity.
- EMD IMF-LZ: about 2 s/window, too slow for the real-time target.

So the bulk build writes 16 cheap features (~9 ms/window). Sample entropy (~55 ms/window)
is opt-in via `--spen` / `spen=True`. The signal is converted to millivolts by `int / 200`
and vftx's own frequency filtering is off (the record is already filtered).

In [4]:
from vfta import features

print('feature columns:', features.feature_names())
print('with sample entropy:', features.feature_names(spen=True)[-1])

# build with features (already wired into build_record/build_database):
# build.build_database('mitdb', window_sec=8.0, jobs=-1)            # 16 cheap features
# build.build_database('mitdb', window_sec=8.0, jobs=-1, spen=True) # + sample entropy

feature columns: ['tcsc', 'tci', 'ste', 'mea', 'psr', 'hilb', 'vf_leak', 'm', 'a2', 'fm', 'lz', 'mav', 'count1', 'count2', 'count3', 'amplitude']
with sample entropy: spen


## Phase 3: load and summarise the dataset

Load every per-record TSV into one DataFrame per database, at both window lengths (8 s and
4 s), and attach the Rhythm / Shock targets. Then summarise: which episode types appear in
how many windows, and the rhythm-class composition.

The tables below load every database in full to show the raw picture (including ahadb, which
carries only beat annotations and a few VF brackets). The study dataset further down applies
the benchmark labelling and restricts ahadb to its 8-series records.

In [5]:
from vfta.segment import LABELS

def load_windows(win):
    return {db: labels.assign_targets(labels.load_database(db, win), win) for db in DBS}
pass #def

def presence_table(data):
    # windows in which each episode type is present (duration > 0), per database
    p = pd.DataFrame({db: {lab: int((df[lab] > 0).sum()) for lab in LABELS}
                      for db, df in data.items()})
    p['ALL'] = p.sum(axis=1)
    p.loc['* windows *'] = {**{db: len(df) for db, df in data.items()},
                            'ALL': sum(len(df) for df in data.values())}
    return p
pass #def

def rhythm_table(data):
    c = pd.DataFrame({db: df['Rhythm'].value_counts() for db, df in data.items()}).fillna(0).astype(int)
    c['ALL'] = c.sum(axis=1)
    return c
pass #def

data = {8.0: load_windows(8.0), 4.0: load_windows(4.0)}
for win, d in data.items():
    print(f'{int(win)} s: {sum(len(df) for df in d.values()):,} windows')
pass #for

8 s: 291,431 windows
4 s: 292,167 windows


### Episode presence: windows containing each rhythm

In [6]:
print('8 s'); display(presence_table(data[8.0]))
print('4 s'); display(presence_table(data[4.0]))

8 s


,mitdb,cudb,vfdb,ahadb,ALL
NSR,65827,1430,18164,0,85421
BGM,3934,0,597,0,4531
TGM,1782,0,0,0,1782
VTH,668,42,5642,0,6352
VFL,182,0,1217,0,1399
VFN,180,4061,0,5549,9790
VFB,0,337,1750,0,2087
AFL,1085,0,0,0,1085
AFB,8488,0,3133,0,11621
EPX,11762,583,18340,0,30685


4 s


,mitdb,cudb,vfdb,ahadb,ALL
NSR,64976,1414,17849,0,84239
BGM,3315,0,553,0,3868
TGM,1473,0,0,0,1473
VTH,448,38,5479,0,5965
VFL,166,0,1022,0,1188
VFN,164,3933,0,5533,9630
VFB,0,337,1722,0,2059
AFL,941,0,0,0,941
AFB,8282,0,3123,0,11405
EPX,11314,575,18033,0,29922


### Rhythm-class composition

In [7]:
print('8 s'); display(rhythm_table(data[8.0]))
print('4 s'); display(rhythm_table(data[4.0]))

8 s


,mitdb,cudb,vfdb,ahadb,ALL
Rhythm,,,,,
MIX,5215,12394,2476,136121,156206
NSR,60557,1118,16257,0,77932
OTHER,20335,522,20456,0,41313
VF,4,3490,1628,5447,10569
VFL,105,0,369,0,474
VT,88,11,4838,0,4937


4 s


,mitdb,cudb,vfdb,ahadb,ALL
Rhythm,,,,,
MIX,3076,12377,1498,136404,153355
NSR,61781,1108,16661,0,79550
OTHER,21419,529,20857,0,42805
VF,2,3646,1659,5480,10787
VFL,124,0,468,0,592
VT,94,15,4969,0,5078


### Study dataset

`load_dataset` applies the benchmark labelling: a window is SHOCK when shockable episodes
(VT/VFL/VF) cover at least 90% of it, NON when no shockable episode is present, and MIX only
when a shockable episode partially straddles the window. An unannotated background window is
therefore non-shockable, which is how these algorithms are scored. ahadb is kept to its
8-series ventricular records (see `labels.RECORD_FILTER`). This is the dataset the feature
screen and shootout use. Under this rule the cudb lead-in and ahadb background become NON, and
MIX shrinks to just transition windows.

In [8]:
for win in (8.0, 4.0):
    study = labels.load_dataset(DBS, win)
    print(f'{int(win)} s study dataset: {len(study):,} windows')
    display(pd.crosstab(study['DB'], study['Shock']))
pass #for

8 s study dataset: 167,783 windows


Shock,MIX,NON,SHOCK
DB,,,
ahadb,102,12371,5447
cudb,590,13432,3513
mitdb,594,85469,241
vfdb,1772,37417,6835


4 s study dataset: 168,243 windows


Shock,MIX,NON,SHOCK
DB,,,
ahadb,53,12427,5480
cudb,307,13704,3664
mitdb,364,85889,243
vfdb,1127,37889,7096


## Phase 3: feature screen

Score each of the 27 features against the shockable label: point-biserial correlation,
mutual information, single-feature AUC. Flag redundancy with a feature-feature correlation
heatmap. This confirms the two added candidates (HILB, MEA).

In [9]:
# TODO: per-feature correlation / mutual information / AUC ranking

## Phase 3: candidate shootout

Five detectors (TCSC, VFLEAK, SPEC, HILB, MEA): discrimination plus a rough compute cost per
window, at 8 s, leaders rerun at 4 s. The winner is chosen by discrimination weighed against
cost, not the top score alone.

In [10]:
# TODO: candidate detectors (threshold + decision) and the discrimination-vs-cost table

## Phase 4: winner tuning

Sweep the winning feature's threshold, build the ROC curve, pick operating points; report
F1, Se, Sp, PPV, Acc, G-Mean under the three VFL configurations and both windows.

In [11]:
# TODO: ROC sweep and operating-point table for the winner (hypothesis: TCSC)

## Phase 4: flutter vs fibrillation (winner only)

Whether the winning feature separates VFL from VF; if not, fall back to the IMF-LZ features
(LZ on EMD modes), which Hong added for the VF-vs-VT distinction.

In [12]:
# TODO: VFL-vs-VF separation for the winning feature